# CatVTON — LoRA Fine-Tuning on Kaggle T4 x2
### `train_lora_modified.py` — mask-based pipeline

**Before running:**
1. Accelerator → **GPU T4 x2**
2. Settings → **Internet ON**
3. Upload your dataset zip as a Kaggle Dataset (see §3)

Run cells top-to-bottom once per session.

---
## §1 — Verify GPUs

In [ ]:
import torch

assert torch.cuda.is_available(), 'No GPU detected — enable T4 x2 in Kaggle accelerator settings!'

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {props.name}  |  {props.total_memory / 1024**3:.1f} GB')

print(f'\nPyTorch: {torch.__version__}  |  CUDA: {torch.version.cuda}')
print(f'Total GPUs: {torch.cuda.device_count()} — need 2 for T4 x2')

---
## §2 — Install dependencies
> Kaggle already has a CUDA-enabled PyTorch. We skip reinstalling torch to avoid replacing it with a CPU build.

In [ ]:
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise RuntimeError(f'Command failed: {cmd}')
    print(result.stdout[-2000:] or '(done)')

packages = [
    'diffusers==0.29.2',
    'transformers==4.46.3',
    'accelerate==0.33.0',
    'peft==0.12.0',
    'huggingface_hub>=0.24.0,<2.0',
    'safetensors',
    'numpy==1.26.4',
    'pillow==10.3.0',
    'opencv-python-headless==4.10.0.84',
    'scipy==1.13.1',
    'scikit-image==0.24.0',
    'fvcore==0.1.5.post20221221',
    'cloudpickle==3.0.0',
    'omegaconf==2.3.0',
    'pycocotools==2.0.8',
    'PyYAML==6.0.1',
    'tqdm==4.66.4',
    'matplotlib==3.9.1',
]

run('pip install -q ' + ' '.join(f'"{p}"' for p in packages))
print('All packages installed.')

---
## §3 — Clone repo & set working directory

In [ ]:
import os

REPO_URL = 'https://github.com/Ahmed3280/LoRA_Fashion.git'
BRANCH   = 'edited'
REPO_DIR = '/kaggle/working/LoRA_Fashion'

if not os.path.exists(REPO_DIR):
    run(f'git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}')
    print('Repo cloned.')
else:
    run(f'git -C {REPO_DIR} fetch origin')
    run(f'git -C {REPO_DIR} reset --hard origin/{BRANCH}')
    print('Repo updated.')

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

---
## §4 — Point to dataset

Your dataset is already mounted by Kaggle at `/kaggle/input/`.  
Just verify the paths are correct below — no extraction needed.

In [ ]:
from pathlib import Path

# Your Kaggle dataset is already mounted — just set the root
# The folder structure expected: DATA_ROOT/train/image, cloth, agnostic-mask, train_pairs.txt
DATA_ROOT = '/kaggle/input/datasets/ahmedakram1/lora-train'

# Sanity check
required = [
    Path(DATA_ROOT) / 'train' / 'image',
    Path(DATA_ROOT) / 'train' / 'cloth',
    Path(DATA_ROOT) / 'train' / 'agnostic-mask',
    Path(DATA_ROOT) / 'train' / 'train_pairs.txt',
]
all_ok = True
for p in required:
    status = 'OK' if p.exists() else 'MISSING'
    if not p.exists(): all_ok = False
    print(f'  [{status}]  {p}')

if all_ok:
    pairs = (Path(DATA_ROOT) / 'train' / 'train_pairs.txt').read_text().strip().splitlines()
    print(f'\nTotal pairs: {len(pairs)} — ready to train!')
else:
    print('\nFix missing paths before continuing.')

---
## §5 — Hyperparameters
**Edit this cell only.** Everything below reads these variables.

> **T4 note:** T4 does NOT support bf16. `fp16` is set for you — do not change it.

In [ ]:
# ── Model checkpoints ────────────────────────────────────────────────────────
BASE_CKPT           = 'booksforcharlie/stable-diffusion-inpainting'
ATTN_CKPT           = 'zhengchong/CatVTON'
ATTN_CKPT_VERSION   = 'mix'          # 'mix' | 'vitonhd' | 'dresscode'

# ── Output ───────────────────────────────────────────────────────────────────
OUTPUT_DIR          = '/kaggle/working/output/lora_mena'

# ── Image resolution ─────────────────────────────────────────────────────────
# 512x384 = safe for T4 (16 GB per GPU)
# 768x576 = usually fits with grad checkpointing
# 1024x768 = tight (~14 GB), may OOM — try only if 768x576 works
HEIGHT              = 512
WIDTH               = 384

# ── Training ─────────────────────────────────────────────────────────────────
BATCH_SIZE          = 1              # per GPU (2 GPUs → effective 2 per step)
GRAD_ACCUM_STEPS    = 4
NUM_EPOCHS          = 20
MIXED_PRECISION     = 'fp16'         # T4 does NOT support bf16 — keep as fp16
GRADIENT_CKPT       = True

# ── Optimiser ────────────────────────────────────────────────────────────────
LR                  = 1e-4
LR_SCHEDULER        = 'cosine'
LR_WARMUP_STEPS     = 100

# ── LoRA ─────────────────────────────────────────────────────────────────────
LORA_RANK           = 16
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.05
LORA_TARGET_MODULES = 'to_q,to_k,to_v,to_out.0'

# ── Validation / early stopping ───────────────────────────────────────────────
VAL_SPLIT           = 0.1
VAL_SEED            = 42
EVAL_EVERY          = 1
EARLY_STOP_PATIENCE = 5
EARLY_STOP_MIN_DELTA= 0.0

# ── Resume from checkpoint (optional) ────────────────────────────────────────
# Set RESUME_CKPT to the path of your downloaded LoRA weights to continue
# training from where you left off.  Leave as None for a fresh start.
RESUME_CKPT         = None   # e.g. '/kaggle/input/my-lora-weights/best'
START_EPOCH         = 0      # last completed epoch (0 = fresh start)
RESUME_BEST_VAL     = float('inf')  # best val_loss before the crash

# ── Misc ─────────────────────────────────────────────────────────────────────
CLOTH_TYPE          = 'overall'
SAVE_EVERY          = 5
NUM_WORKERS         = 2
SEED                = 42

print('Hyperparameters set.')
print(f'  Resolution           : {WIDTH}x{HEIGHT}')
print(f'  Mixed precision      : {MIXED_PRECISION}  (fp16 required for T4)')
print(f'  Effective batch size : {BATCH_SIZE} per GPU × 2 GPUs × {GRAD_ACCUM_STEPS} accum = {BATCH_SIZE*2*GRAD_ACCUM_STEPS}')
print(f'  LoRA rank/alpha      : {LORA_RANK}/{LORA_ALPHA}')
print(f'  Epochs               : {NUM_EPOCHS}')
if RESUME_CKPT:
    print(f'  RESUMING from        : {RESUME_CKPT}  (epoch {START_EPOCH}, best_val={RESUME_BEST_VAL})')

---
## §6 — Configure Accelerate for 2 GPUs

In [ ]:
import yaml, os

NUM_GPUS = max(1, __import__('torch').cuda.device_count())
print(f'Detected {NUM_GPUS} GPU(s) — configuring accelerate for {NUM_GPUS} process(es).')

acc_cfg = {
    'compute_environment': 'LOCAL_MACHINE',
    'debug': False,
    'distributed_type': 'MULTI_GPU' if NUM_GPUS > 1 else 'NO',
    'downcast_bf16': 'no',
    'gpu_ids': 'all',
    'machine_rank': 0,
    'main_training_function': 'main',
    'mixed_precision': MIXED_PRECISION,
    'num_machines': 1,
    'num_processes': NUM_GPUS,
    'rdzv_backend': 'static',
    'same_network': True,
    'tpu_env': [],
    'tpu_use_cluster': False,
    'tpu_use_sudo': False,
    'use_cpu': False,
}

cfg_dir = os.path.expanduser('~/.cache/huggingface/accelerate')
os.makedirs(cfg_dir, exist_ok=True)
with open(os.path.join(cfg_dir, 'default_config.yaml'), 'w') as f:
    yaml.dump(acc_cfg, f)

print(f'Accelerate config written — distributed_type: {acc_cfg["distributed_type"]}')

---
## §7 — Pre-download model weights

In [ ]:
import os
os.environ['HF_HUB_HTTP_TIMEOUT'] = '120'

from huggingface_hub import snapshot_download

print('Downloading base SD-inpainting model...')
snapshot_download(repo_id=BASE_CKPT)
print('Done.')

print('Downloading CatVTON attention checkpoint...')
snapshot_download(repo_id=ATTN_CKPT)
print('Done.')

print('Downloading VAE...')
snapshot_download(repo_id='stabilityai/sd-vae-ft-mse')
print('All weights cached.')

---
## §8 — VRAM check

In [ ]:
import torch

for i in range(torch.cuda.device_count()):
    vram = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f'GPU {i}: {vram:.1f} GB')

# Per-GPU estimate (each GPU runs its own batch in DDP)
pixels = HEIGHT * WIDTH
if pixels <= 512*384:
    est, label = 8, '512×384'
elif pixels <= 768*576:
    est, label = 12, '768×576'
else:
    est, label = 15, '1024×768'

vram_0 = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'\nEstimated peak VRAM per GPU at {label} + fp16 + grad-ckpt: ~{est} GB')
if vram_0 < est:
    print(f'WARNING: {vram_0:.1f} GB < {est} GB — lower HEIGHT/WIDTH')
else:
    print(f'OK: {vram_0:.1f} GB available, ~{est} GB needed — should fit.')

---
## §9 — Build command & launch training

In [ ]:
import torch

NUM_GPUS = torch.cuda.device_count()
no_grad_ckpt_flag = '' if GRADIENT_CKPT else '--no_gradient_checkpointing'

cmd_parts = [
    f'accelerate launch --num_processes {NUM_GPUS}',
    'train_lora_modified.py',
    f'--data_root            "{DATA_ROOT}"',
    f'--base_ckpt            "{BASE_CKPT}"',
    f'--attn_ckpt            "{ATTN_CKPT}"',
    f'--attn_ckpt_version    {ATTN_CKPT_VERSION}',
    f'--output_dir           "{OUTPUT_DIR}"',
    f'--cloth_type           {CLOTH_TYPE}',
    f'--height               {HEIGHT}',
    f'--width                {WIDTH}',
    f'--batch_size           {BATCH_SIZE}',
    f'--gradient_accumulation_steps {GRAD_ACCUM_STEPS}',
    f'--num_epochs           {NUM_EPOCHS}',
    f'--mixed_precision      {MIXED_PRECISION}',
    f'--lr                   {LR}',
    f'--lr_scheduler         {LR_SCHEDULER}',
    f'--lr_warmup_steps      {LR_WARMUP_STEPS}',
    f'--lora_rank            {LORA_RANK}',
    f'--lora_alpha           {LORA_ALPHA}',
    f'--lora_dropout         {LORA_DROPOUT}',
    f'--lora_target_modules  "{LORA_TARGET_MODULES}"',
    f'--val_split            {VAL_SPLIT}',
    f'--val_seed             {VAL_SEED}',
    f'--eval_every           {EVAL_EVERY}',
    f'--early_stop_patience  {EARLY_STOP_PATIENCE}',
    f'--early_stop_min_delta {EARLY_STOP_MIN_DELTA}',
    f'--save_every           {SAVE_EVERY}',
    f'--num_workers          {NUM_WORKERS}',
    f'--seed                 {SEED}',
    no_grad_ckpt_flag,
]

# Append resume flags only when RESUME_CKPT is set
if RESUME_CKPT:
    cmd_parts += [
        f'--resume_from_checkpoint "{RESUME_CKPT}"',
        f'--start_epoch          {START_EPOCH}',
        f'--resume_best_val      {RESUME_BEST_VAL}',
    ]

TRAIN_CMD = ' '.join(p for p in cmd_parts if p.strip())
print('Training command:')
print(TRAIN_CMD)

In [ ]:
import subprocess, os

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.environ['HF_HUB_HTTP_TIMEOUT'] = '120'

process = subprocess.Popen(
    TRAIN_CMD,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

if process.returncode == 0:
    print('\nTraining finished successfully.')
else:
    print(f'\nTraining exited with code {process.returncode}')

---
## §10 — Inspect & compress output

Kaggle keeps `/kaggle/working/` between cells but **not after the session ends**.
Compress the LoRA weights so you can download them before the session expires.

In [ ]:
from pathlib import Path
import shutil

out = Path(OUTPUT_DIR)
if not out.exists():
    print('Output directory not found — training may not have run.')
else:
    checkpoints = sorted(out.iterdir())
    print(f'Checkpoints in {OUTPUT_DIR}:')
    for ckpt in checkpoints:
        if ckpt.is_dir():
            files = list(ckpt.iterdir())
            print(f'  {ckpt.name}/  ({len(files)} files)')
        else:
            print(f'  {ckpt.name}')

    # Compress the final checkpoint for download
    final = out / 'final'
    zip_out = '/kaggle/working/lora_mena_final'
    if final.exists():
        print(f'\nCompressing {final} ...')
        shutil.make_archive(zip_out, 'zip', final)
        zip_size = Path(zip_out + '.zip').stat().st_size / 1024**2
        print(f'Saved: {zip_out}.zip  ({zip_size:.1f} MB)')
        print('Download it from the Kaggle output panel on the right.')
    else:
        print(f'\nNo final/ checkpoint yet — training may still be running or was interrupted.')